# Notebook 12 – Train, Validation & Test Data

## 1. Dataset Roles
- **Training set:** fits model parameters.
- **Validation set:** supports model selection and hyperparameter tuning.
- **Test set:** final evaluation after decisions are complete.

## 2. Train-Test Split
Split before fitting preprocessing or a model. `random_state` makes a random split reproducible. `stratify=y` helps preserve class proportions for classification.

In [1]:
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X_array, y_array = make_classification(
    n_samples=500, n_features=5, n_informative=3, n_redundant=0,
    weights=[0.8, 0.2], random_state=42
)
X = pd.DataFrame(X_array, columns=[f"feature_{i}" for i in range(1, 6)])
y = pd.Series(y_array, name="target")
print("Features shape:", X.shape)
print("Target proportions:\n", y.value_counts(normalize=True).round(2))

Features shape: (500, 5)
Target proportions:
 target
0    0.79
1    0.21
Name: proportion, dtype: float64


## 3. Train-Test Split

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (400, 5) (400,)
Test: (100, 5) (100,)


## 4. Train-Validation-Test Split
This example creates approximately 60% train, 20% validation, and 20% test. The test set is held out first; then 25% of the remaining 80% is used for validation.

In [3]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)
print("Train:", len(X_train), "Validation:", len(X_val), "Test:", len(X_test))

Train: 300 Validation: 100 Test: 100


## 5. Stratified Sampling
Stratification keeps class proportions approximately similar across splits, which is useful for imbalanced classification.

In [4]:
for name, labels in [("All", y), ("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    print(name, labels.value_counts(normalize=True).round(3).to_dict())

All {0: 0.794, 1: 0.206}
Train {0: 0.797, 1: 0.203}
Validation {0: 0.79, 1: 0.21}
Test {0: 0.79, 1: 0.21}


## 6. Data Leakage
Leakage occurs when information that would not legitimately be available at prediction time influences training or evaluation.

### Incorrect preprocessing workflow
Fit the scaler on all rows before splitting. Test-set distribution information influences the learned scaling parameters.

In [5]:
from sklearn.preprocessing import StandardScaler

# INCORRECT: fit on the entire dataset before splitting
scaler_bad = StandardScaler()
X_scaled_bad = scaler_bad.fit_transform(X)
X_train_bad, X_test_bad, y_train_bad, y_test_bad = train_test_split(
    X_scaled_bad, y, test_size=0.2, random_state=42
)
print("Incorrect: scaler learned from train and test rows.")

Incorrect: scaler learned from train and test rows.


### Correct preprocessing workflow
Split first. Fit preprocessing only on training data, then transform validation/test data with the already fitted object.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Train shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)

Train shape: (400, 5)
Test shape: (100, 5)


## 7. Pipeline
A pipeline helps keep preprocessing and modelling together and prevents fitting preprocessing on held-out data during cross-validation.

In [7]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Test accuracy:", round(accuracy_score(y_test, pred), 3))

Test accuracy: 0.93


## 8. Temporal Splitting
For time-dependent data, train on earlier records and test on later records. Random splitting can let future patterns influence evaluation. For example, train on January–September and test on October–December.

## 9. Key Rules
1. Split before fitting preprocessing.
2. Fit imputers, scalers, encoders, feature selectors, and resamplers on training data only.
3. Use validation data for model selection; keep test data for final evaluation.
4. Use stratification for many classification problems.
5. Use chronological splitting for temporal prediction.